## Ch10-01 -- A real traceability graph, and one real, unjustified widget

This notebook introduces a real traceability graph over the model's two named requirement usages; after running it you can tell, for each one, what functional intent it expresses, what allocation and realization carry it forward, and what verification evidence, if any, actually exists, and you will have found one of this tutorial's own strongest pieces of formal evidence tied to no requirement at all.

SEBoK defines traceability as "the degree to which a relationship can be established between two or more products of the development process" (`uv run python -m glossary tutorial traceability`); Douglas's own story names what it is for: "we're left with a traceability map that connects the as-designed system with the requirements," used to audit missed requirements, unjustified widgets, and to drive verification tests. Chapter 9's own coverage report joined one pair of surfaces (`RequirementUsage`, `SatisfyRequirementUsage`) for one question: has a positive claim ever been made? This notebook extends that single join into the full chain Douglas's own idea names: from a requirement's own functional intent, through whatever allocation and realization carry it forward, to the verification evidence that actually exists, built from real queries (`model.query()`, `get_satisfy_relationships()`, `requirement_coverage()`, `allocations_for()`, `supertypes_transitively()`), not a hand-typed table. This chapter adds no new named model element: `models/ch10-cumulative.sysml` carries `models/ch08-cumulative.sysml`'s content forward unchanged, the same design choice Chapter 9 made (see this chapter's own [index.md](index.md) for why, unlike Chapter 9, this chapter still commits its own fixture file).

In [1]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch10-cumulative.sysml").read_text()
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"


The model loads cleanly. Before tracing anything through it, the same language-tier control every chapter carries, chosen here to match what this chapter actually traces: an `allocate` naming a feature that was never declared still fails to load, which matters directly for a chapter about following allocation and realization chains -- a broken link can never silently stand in for a real one.

In [2]:
# An allocate naming an undeclared feature fails to load; it can never silently
# stand in for a real allocation link this chapter's own traceability graph follows.
bad_source = """
package Bad {
    action def A;
    part def W { perform action a : A; }
    part w : W;
    allocation badAlloc allocate A::missing to w;
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok, "Expected failure for an allocation naming an undeclared feature"
print(f"Negative control ok: bad.ok={bad.ok}")


Negative control ok: bad.ok=False


With the model loaded, the graph starts from the model's two named requirement usages and, for each, the requirement definition's own declared `subject` feature: not read off the source text by eye, but found the same way any other query in this tutorial finds a real fact, by reading each candidate feature's own `sysx:sourceText` in the API-JSON export for the literal `subject` keyword SysML v2 itself requires there (SysML v2 formal/2026-03-02 SS8.3, RequirementDefinition).

In [3]:
from toaster.query import ApiIndex, find_requirements

idx = ApiIndex(model)


def requirement_subject(idx, req_def_qn):
    """The (name, type) of req_def_qn's own declared `subject` feature, found by
    reading each of its owned ReferenceUsage features for the literal `subject`
    keyword in its own sysx:sourceText, not assumed from the requirement's name."""
    for e in idx.of_type("ReferenceUsage"):
        qn = e.get("qualifiedName", "")
        if qn.startswith(req_def_qn + "::") and e.get("sysx:sourceText", "").strip().startswith("subject "):
            return e["declaredName"], idx.type_names(qn)[0]
    return None, None


requirements = sorted(r.id for r in find_requirements(model))
for r in requirements:
    req_def = idx.type_names(r)[0]
    subject_name, subject_type = requirement_subject(idx, req_def)
    print(f"{r}: def={req_def}  subject={subject_name} : {subject_type}")


ToasterDemo::heatGenerationReq: def=ToasterDemo::HeatGenerationReq  subject=heatGen : ToasterDemo::HeatGenerator
ToasterDemo::timely: def=ToasterDemo::TimelyToast  subject=toaster : ToasterDemo::Toaster


`heatGenerationReq`'s own subject is `heatGen : HeatGenerator`, an engineering rating on the logical carrier one level below the heating system (Chapter 6). `timely`'s own subject is `toaster : Toaster`, the toaster as a whole. Both are real functional intents the requirement definitions themselves state; the next cells trace what carries each one forward.

In [4]:
from toaster.query import allocations_for, supertypes_transitively

heatgen_allocations = allocations_for(model, "ToasterDemo::HeatingAssembly::heatGen", inherit=True, index=idx)
heatgen_realizers = supertypes_transitively(model, "ToasterDemo::ResistanceCoil")
print("heatGenerationReq's subject is allocated from:")
for a in heatgen_allocations:
    print(f"  {a}")
print(f"ResistanceCoil (the realizer chosen for HeatGenerator) itself specializes: {heatgen_realizers}")


heatGenerationReq's subject is allocated from:
  {'id': 'ToasterDemo::heatGenAllocation', 'type': 'AllocationUsage', 'ends': [['ToasterDemo::ApplyHeat::generateHeat'], ['ToasterDemo::HeatingAssembly::heatGen']]}
ResistanceCoil (the realizer chosen for HeatGenerator) itself specializes: {'ToasterDemo::HeatGenerator'}


`heatGenAllocation` allocates `ApplyHeat::generateHeat`, the functional action Chapter 4 defined, to `HeatingAssembly::heatGen`, the logical carrier; `ResistanceCoil` specializes `HeatGenerator` (Chapter 6's own mechanism selection, `AS-C06`), and `rated`/`weak` are its two physical candidates. The same pattern, traced for `timely`'s own subject next.

`Toaster` itself, unlike `HeatGenerator`, has no further physical subtype: its own candidates are usages typed directly by it. `specializes_transitively()` reports every usage that specializes or is typed by `Toaster`, which also catches the two requirement definitions' own internal `subject` placeholders (`TimelyToast::toaster`, `TimelyToastTest::toaster`, nested inside the requirement definitions themselves, not real candidates); filtering to elements owned directly by the top-level package leaves the two real ones.

In [5]:
from toaster.query import specializes_transitively

toaster_allocations = allocations_for(model, "ToasterDemo::Toaster::heating", inherit=True, index=idx)
toaster_candidates_raw = specializes_transitively(model, "ToasterDemo::Toaster")
toaster_candidates = sorted(
    qn for qn in toaster_candidates_raw if idx.by_qn[qn].get("owner", {}).get("@id") == "ToasterDemo"
)
print("timely's subject is allocated from:")
for a in toaster_allocations:
    print(f"  {a}")
print(f"specializes_transitively(Toaster), raw: {sorted(toaster_candidates_raw)}")
print(f"filtered to real, top-level candidates: {toaster_candidates}")


timely's subject is allocated from:
  {'id': 'ToasterDemo::heatAllocation', 'type': 'AllocationUsage', 'ends': [['ToasterDemo::ToastBread::applyHeat'], ['ToasterDemo::Toaster::heating']]}
specializes_transitively(Toaster), raw: ['ToasterDemo::TimelyToast::toaster', 'ToasterDemo::TimelyToastTest::toaster', 'ToasterDemo::nominal', 'ToasterDemo::slow']
filtered to real, top-level candidates: ['ToasterDemo::nominal', 'ToasterDemo::slow']


`heatAllocation` allocates `ToastBread::applyHeat` to `Toaster::heating`; `Toaster`'s own real candidates are `nominal` and `slow`. Both chains reach real physical candidates. What verification evidence actually exists for each is the last, and most consequential, link.

In [6]:
from toaster.query import requirement_coverage

coverage = {c["requirement"]: c for c in requirement_coverage(model, idx)}
for r in requirements:
    print(coverage[r])


{'requirement': 'ToasterDemo::heatGenerationReq', 'satisfied_by': ['ToasterDemo::rated'], 'failed_by': ['ToasterDemo::weak'], 'covered': True}
{'requirement': 'ToasterDemo::timely', 'satisfied_by': [], 'failed_by': ['ToasterDemo::slow'], 'covered': False}


`heatGenerationReq` is covered on both sides: `rated` really satisfies it, `weak` really fails it, and both claims are genuine positive/negative evidence about a real candidate. `timely` is not covered: the only claim against it is negative (`slow` fails it), and `TimelyToastTest`'s own `verify timely;` objective names the requirement without binding any subject (Chapter 9's own finding, reproduced here directly against the same real model). This is an absence of a claim, not evidence that `nominal` fails `timely`: `Toaster::cycleTime` is still a settable attribute, not derived from anything, so no one has ever actually checked whether `nominal` satisfies `timely` in the first place.

In [7]:
traceability_graph = [
    {
        "requirement": "ToasterDemo::heatGenerationReq",
        "functional_intent": "heatGen : HeatGenerator (a heat generator rated for at least 600 W)",
        "allocation": "heatGenAllocation: ApplyHeat::generateHeat -> HeatingAssembly::heatGen",
        "realization": "ResistanceCoil (:> HeatGenerator); candidates rated, weak",
        "verification_evidence": (
            f"satisfied_by={coverage['ToasterDemo::heatGenerationReq']['satisfied_by']}, "
            f"failed_by={coverage['ToasterDemo::heatGenerationReq']['failed_by']} "
            "(bidirectional: a real positive claim AND a real negative claim)"
        ),
    },
    {
        "requirement": "ToasterDemo::timely",
        "functional_intent": "toaster : Toaster (a toasting cycle completing in at most 180 s)",
        "allocation": "heatAllocation: ToastBread::applyHeat -> Toaster::heating",
        "realization": "Toaster itself; candidates nominal, slow",
        "verification_evidence": (
            f"satisfied_by={coverage['ToasterDemo::timely']['satisfied_by']} (none), "
            f"failed_by={coverage['ToasterDemo::timely']['failed_by']} "
            "(one-sided: only a negative claim, plus an unbound verify objective)"
        ),
    },
]
for row in traceability_graph:
    print(row["requirement"] + ":")
    for key, value in row.items():
        if key != "requirement":
            print(f"  {key}: {value}")


ToasterDemo::heatGenerationReq:
  functional_intent: heatGen : HeatGenerator (a heat generator rated for at least 600 W)
  allocation: heatGenAllocation: ApplyHeat::generateHeat -> HeatingAssembly::heatGen
  realization: ResistanceCoil (:> HeatGenerator); candidates rated, weak
  verification_evidence: satisfied_by=['ToasterDemo::rated'], failed_by=['ToasterDemo::weak'] (bidirectional: a real positive claim AND a real negative claim)
ToasterDemo::timely:
  functional_intent: toaster : Toaster (a toasting cycle completing in at most 180 s)
  allocation: heatAllocation: ToastBread::applyHeat -> Toaster::heating
  realization: Toaster itself; candidates nominal, slow
  verification_evidence: satisfied_by=[] (none), failed_by=['ToasterDemo::slow'] (one-sided: only a negative claim, plus an unbound verify objective)


`heatGenerationReq` traces all the way from a stated functional intent through a real allocation and a real mechanism selection to genuine, opposite-polarity verification evidence. `timely` traces just as far through intent, allocation and realization, but stops one link short: no candidate has ever been positively checked against it. Both are real findings about the model as it actually stands, not a contrast built to make one requirement look better than the other.

One more real finding this graph surfaces, in the other direction: does the model's own strongest formal evidence trace back to any requirement at all? `deliveredEnergyBoundedBySupply` (Chapter 8) is the one property in this whole tutorial proved by Z3 for every value its unbound features admit, not merely evaluated at one point. Searched directly against the same API-JSON export every other query in this notebook uses: is it named as the `subsets` of any `SatisfyRequirementUsage`, anywhere in the model?

In [8]:
satisfy_targets = {
    s["subsets"]["@id"] for s in idx.of_type("SatisfyRequirementUsage") if "subsets" in s
}
delivered_energy_bounded = idx.by_qn["ToasterDemo::deliveredEnergyBoundedBySupply"]
tied_to_a_requirement = delivered_energy_bounded["@id"] in satisfy_targets

print(f"deliveredEnergyBoundedBySupply @type: {delivered_energy_bounded['@type']}")
print(f"Ids ever named as a satisfy's own requirement: {sorted(satisfy_targets)}")
print(f"deliveredEnergyBoundedBySupply tied to any requirement usage: {tied_to_a_requirement}")
assert not tied_to_a_requirement


deliveredEnergyBoundedBySupply @type: AssertConstraintUsage
Ids ever named as a satisfy's own requirement: ['ToasterDemo__heatGenerationReq', 'ToasterDemo__timely']
deliveredEnergyBoundedBySupply tied to any requirement usage: False


It is not. `deliveredEnergyBoundedBySupply` is an `AssertConstraintUsage`, not a `RequirementUsage`, and its own id never appears as any `SatisfyRequirementUsage`'s target. This is a real, honest finding about this tutorial's own accumulated model, not a manufactured one, and Douglas's own traceability concern names exactly this failure mode: a design element ("widget") with no requirement to justify it. Here the shape is reversed but the concern is the same, a genuine piece of verification evidence with no requirement to justify *it*: the strongest formal proof this tutorial has built connects to no stated need at all. `Toaster::cycleTime` is not derived from anything and `deliveredEnergyBoundedBySupply` is not tied to any requirement usage are two different gaps in the same graph; neither is fixed here (a non-goal of this chapter), and neither should be papered over by forcing a connection the model does not actually make.

The graph above, and the search that follows it, say the same thing two different ways: a traceability graph is not just a map of what connects, it is just as much a map of what does not, and both kinds of gap are real findings, not defects in the query.

Try the chapter exercise in `exercises/ch10/exercise.ipynb`: it asks you to extend the traceability graph built above to include the bread-handling allocation links.